In [ ]:
import json
import pandas as pd

DATA = 'data/cymo'
METADATA = 'data/final_depression_full_metadata.jsonl'

INTERVAL_DAYS = 30
MIN_WORDS = 500        # per interval
MIN_POSTS = 3          # per interval
MIN_INTERVALS = 8      # valid intervals per participant
MIN_SPAN_DAYS = 180    # first to last valid interval
MIN_PAIRS = 3          # successive valid pairs, for instability, persistence and transitions

In [ ]:
features = list(json.load(open(f'{DATA}/float64_features.json'))['features'])
measures = pd.read_csv(f'{DATA}/cymo_measures.tsv', sep='\t')

groups = (measures[measures['output_column'].isin(features)]
          .groupby('category')['output_column'].apply(list).to_dict())
{g: len(cols) for g, cols in groups.items()}

In [ ]:
sentences = pd.read_parquet(f'{DATA}/final_depression_cymo_sentences.parquet',
                            columns=['user_id', 'post_id'] + features)
posts = sentences.groupby(['user_id', 'post_id'], sort=False)[features].mean().reset_index()

chunks = pd.read_json(METADATA, lines=True, chunksize=500_000, dtype={'user_id': int, 'post_id': str})
meta = pd.concat(chunk[['user_id', 'post_id', 'created_utc', 'word_count']] for chunk in chunks)
posts = posts.merge(meta, on=['user_id', 'post_id'])
posts['date'] = pd.to_datetime(posts['created_utc'], unit='s')
posts.head()

In [ ]:
posts['interval'] = (posts['date'] - posts['date'].min()).dt.days // INTERVAL_DAYS

g = posts.groupby(['user_id', 'interval'])
intervals = g[features].mean().join(g.agg(n_posts=('post_id', 'size'), n_words=('word_count', 'sum')))
valid = intervals[(intervals['n_words'] >= MIN_WORDS) & (intervals['n_posts'] >= MIN_POSTS)]
valid.head()

In [ ]:
interval = valid.index.get_level_values('interval').to_series(index=valid.index)
by_user = interval.groupby(level='user_id')
span_days = (by_user.max() - by_user.min()) * INTERVAL_DAYS
eligible = by_user.size()[(by_user.size() >= MIN_INTERVALS) & (span_days >= MIN_SPAN_DAYS)].index

cohort = valid.loc[valid.index.get_level_values('user_id').isin(eligible), features]
cohort.index.get_level_values('user_id').nunique(), len(cohort)

In [ ]:
x = cohort
interval = x.index.get_level_values('interval').to_series(index=x.index)
user = x.groupby(level='user_id')

# successive = the previous valid interval is the one directly before; a gap breaks the pair
successive = interval.groupby(level='user_id').diff() == 1
n_pairs = successive.groupby(level='user_id').sum()

centered = x - user.transform('mean')
change = user.diff()[successive]
lag_product = (centered * centered.groupby(level='user_id').shift())[successive]
t = interval - interval.groupby(level='user_id').transform('mean')   # elapsed time, gaps kept

variability = user.std()
instability = (change ** 2).groupby(level='user_id').mean()          # MSSD
persistence = (lag_product.groupby(level='user_id').sum().div(n_pairs, axis=0)
               / (centered ** 2).groupby(level='user_id').mean())     # lag-1 autocorrelation
drift = (centered.mul(t, axis=0).groupby(level='user_id').sum()
         .div((t ** 2).groupby(level='user_id').sum(), axis=0))      # OLS slope per interval
transitions = (change.abs() > variability.reindex(change.index, level='user_id')).groupby(level='user_id').mean()  # share of successive changes > 1 SD

enough_pairs = n_pairs >= MIN_PAIRS
dynamics = pd.concat({
    'variability': variability,
    'instability': instability.where(enough_pairs, axis=0),
    'persistence': persistence.where(enough_pairs, axis=0),
    'drift': drift,
    'transitions': transitions.where(enough_pairs, axis=0),
}, axis=1)
dynamics.head()

In [ ]:
dynamics_by_group = {g: dynamics.loc[:, (slice(None), cols)] for g, cols in groups.items()}
dynamics_by_group['Readability'].head()

In [ ]:
import textwrap
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

RESULTS = Path('results')
RESULTS.mkdir(exist_ok=True)

INK, MUTED, GRID = '#0b0b0b', '#52514e', '#e6e5e0'
GROUP_COLORS = dict(zip(groups, ['#2a78d6', '#eb6834', '#1baf7a', '#eda100']))
names = measures.set_index('output_column')['name']

plt.rcParams.update({
    'font.size': 9, 'axes.titlesize': 10, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.edgecolor': MUTED,
    'axes.labelcolor': INK, 'xtick.color': MUTED, 'ytick.color': MUTED,
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.6, 'axes.axisbelow': True,
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
})

def caption(fig, text):
    fig.text(0.01, 0, textwrap.fill(text, int(fig.get_figwidth() * 15)),
             ha='left', va='top', fontsize=8.5, color=MUTED)

In [ ]:
n_valid = cohort.groupby(level='user_id').size()

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), layout='constrained')
axes[0].hist(n_valid, bins=np.arange(MIN_INTERVALS, n_valid.max() + 2) - 0.5,
             color='#2a78d6', edgecolor='white', linewidth=0.5)
axes[0].set(title='Valid intervals per participant', xlabel='Valid 30-day intervals', ylabel='Participants')
axes[1].hist(span_days[eligible] / 365, bins=30, color='#2a78d6', edgecolor='white', linewidth=0.5)
axes[1].set(title='Observation span per participant', xlabel='Years from first to last valid interval', ylabel='Participants')
caption(fig,
    f'Figure 1. Longitudinal data available for the primary cohort ({len(eligible):,} of {posts['user_id'].nunique():,} participants '
    f'with dated depression posts). Posts were binned into {INTERVAL_DAYS}-day intervals; an interval was valid when it '
    f'held at least {MIN_WORDS} words across at least {MIN_POSTS} posts or comments. Participants entered the cohort with '
    f'at least {MIN_INTERVALS} valid intervals spanning at least {MIN_SPAN_DAYS} days. Left: number of valid intervals '
    f'(median {n_valid.median():.0f}). Right: time from the first to the last valid interval '
    f'(median {span_days[eligible].median() / 365:.1f} years); invalid intervals in between are gaps, not filled in.')
fig.savefig(RESULTS / 'fig1_cohort.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
MEASURE = 'FKGL'
example = n_pairs.idxmax()   # the participant with the most successive pairs

series = cohort.loc[example, MEASURE]
series = series.reindex(range(series.index.min(), series.index.max() + 1))   # gaps become NaN
dates = posts['date'].min().normalize() + pd.to_timedelta(series.index * INTERVAL_DAYS, unit='D')
mean, sd, slope = series.mean(), series.std(), dynamics.loc[example, ('drift', MEASURE)]
t = pd.Series(series.index, index=series.index)
change = series.diff()
shift = change.abs() > sd

fig, ax = plt.subplots(figsize=(10, 3.8), layout='constrained')
ax.axhspan(mean - sd, mean + sd, color='#cde2fb', alpha=0.6, lw=0, label='Person mean ± 1 SD (variability)')
ax.plot(dates, mean + slope * (t - t[series.notna()].mean()), color=MUTED, lw=1.5, ls='--', label='Linear trend (drift)')
ax.plot(dates, series, color='#2a78d6', lw=2, marker='o', ms=4, label='Valid interval')
ax.plot(dates[shift], series[shift], 'o', ms=9, mfc='none', mec='#eb6834', mew=2, label='Transition (change > 1 SD)')
ax.set(title=f'{names[MEASURE]} ({MEASURE}) for one participant', ylabel=MEASURE)
ax.legend(loc='upper left', ncols=4, frameon=False, fontsize=8, bbox_to_anchor=(0, -0.08))

m = dynamics.loc[example, (slice(None), MEASURE)].droplevel(1)
caption(fig,
    f'Figure 2. How the five dynamics metrics are read from one trajectory: participant {example}, the cohort member '
    f'with the most successive valid intervals ({n_pairs[example]}). Each dot is the mean {MEASURE} of the posts in a '
    f'valid {INTERVAL_DAYS}-day interval; breaks in the line are invalid intervals, left missing. Variability is the SD '
    f'of the dots (band; {m.variability:.2f}). Instability is the mean squared change between directly adjacent valid '
    f'intervals ({m.instability:.2f}); changes across a gap are not counted. Persistence is the lag-1 autocorrelation over '
    f'those same adjacent pairs ({m.persistence:.2f}). Drift is the least-squares slope over elapsed intervals, gaps '
    f'included (dashed; {m.drift:+.3f} per interval). Transitions are the share of adjacent changes larger than 1 SD '
    f'(circled; {m.transitions:.2f}).')
fig.savefig(RESULTS / 'fig2_example_trajectory.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
per_year = 365 / INTERVAL_DAYS
summary = {
    'Persistence\n(lag-1 autocorrelation)': dynamics['persistence'],
    'Drift\n(person SDs per year)': dynamics['drift'] / dynamics['variability'] * per_year,
    'Transitions\n(share of changes > 1 SD)': dynamics['transitions'],
}
order = [m for g in groups for m in sorted(groups[g], key=lambda m: dynamics['persistence'][m].median())]
y = np.arange(len(order))
color = [GROUP_COLORS[g] for g in groups for _ in groups[g]]

fig, axes = plt.subplots(1, 3, figsize=(11, 10), sharey=True, layout='constrained')
for ax in axes[:2]:
    ax.axvline(0, color=MUTED, lw=0.8)
for ax, (label, values) in zip(axes, summary.items()):
    q = values[order].quantile([0.25, 0.5, 0.75])
    ax.hlines(y, q.loc[0.25], q.loc[0.75], color=color, lw=2)
    ax.scatter(q.loc[0.5], y, c=color, s=30, zorder=3, edgecolor='white', linewidth=1)
    ax.set_title(label)
    ax.grid(axis='y', visible=False)
axes[0].set_yticks(y, [f'{names[m]} ({m})' for m in order], fontsize=7.5)
axes[0].invert_yaxis()
handles = [plt.Line2D([], [], marker='o', ls='', color=c, label=g) for g, c in GROUP_COLORS.items()]
fig.legend(handles=handles, title='CYMO category', loc='outside upper center', ncols=4, frameon=False)

caption(fig,
    f'Figure 3. Temporal properties of each CYMO measure across the cohort ({len(eligible):,} participants). Dot = median '
    f'across participants; line = interquartile range. Persistence and transitions use only directly adjacent valid '
    f'intervals and need at least {MIN_PAIRS} such pairs ({(n_pairs >= MIN_PAIRS).sum():,} participants). Drift is the '
    f'per-person least-squares slope divided by that person\'s SD and scaled to a year, so measures on different scales '
    f'are comparable: 0 means no systematic change. Persistence near 0 means a month\'s value says little about the next. '
    f'Measures are grouped by CYMO category and sorted by median persistence. Variability and instability are '
    f'left out here because they are in each measure\'s own units (see Figure 4).')
fig.savefig(RESULTS / 'fig3_metrics_by_measure.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
metrics = ['variability', 'instability', 'persistence', 'drift', 'transitions']
per_measure = []
for m in features:
    table = dynamics.xs(m, axis=1, level=1)[metrics].assign(drift=lambda d: d['drift'].abs())
    per_measure.append(table.corr(method='spearman'))
corr = pd.concat(per_measure).groupby(level=0).median().loc[metrics, metrics]
corr = corr.where(np.tril(np.ones(corr.shape, bool), k=-1))   # lower triangle, no diagonal

cmap = LinearSegmentedColormap.from_list('diverging', ['#2a78d6', '#f0efec', '#e34948'])
labels = [m.capitalize() if m != 'drift' else '|Drift|' for m in metrics]

fig, ax = plt.subplots(figsize=(6, 5.2), layout='constrained')
im = ax.imshow(corr, cmap=cmap, vmin=-1, vmax=1)
for i in range(len(metrics)):
    for j in range(i):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', fontsize=9, color=INK)
ax.set_xticks(range(len(metrics) - 1), labels[:-1], rotation=30, ha='right')
ax.set_yticks(range(1, len(metrics)), labels[1:])
ax.set_xlim(-0.5, len(metrics) - 1.5)
ax.set_ylim(len(metrics) - 0.5, 0.5)
ax.grid(False)
ax.spines[:].set_visible(False)
ax.set_title('How strongly the metrics overlap')
fig.colorbar(im, ax=ax, shrink=0.8, label='Spearman correlation across participants')
caption(fig,
    f'Figure 4. Overlap between the five dynamics metrics. For each of the {len(features)} CYMO measures, the metrics were '
    f'correlated across participants (Spearman); each cell is the median of those {len(features)} correlations. Drift is '
    f'taken as its absolute value (size of the trend, either direction). Values near 0 mean two metrics capture different '
    f'information; variability and instability are expected to overlap because both grow with how much a person\'s '
    f'language moves.')
fig.savefig(RESULTS / 'fig4_metric_overlap.png', dpi=200, bbox_inches='tight')
plt.show()